In [7]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from models import MLP, SimpleTransformerModel
from datasets import *

['F', 'MLP', 'PluckerTropicalSpace', 'Sequence', 'SimpleTransformerModel', 'TransformerBlock', 'TransformerEncoder', 'TropicalAttention', 'TropicalAttention_', 'TropicalLinear', 'Tuple', 'Union', 'VanillaAttention', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'adaptive_temperature_softmax', 'comb', 'combinations', 'math', 'nn', 'smooth_max', 'smooth_min', 'torch', 'trop_norm', 'tropical_sinkhorn_normalization']


In [8]:
# setup
d = 10

# Generate a batch of random item lists (inputs) and mock ground truth
x_train = torch.rand(1000, d) # [B, d]
y_train = topksubset(x_train, k=3)

model = MLP()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)  # Decay LR every 100 epochs
epochs = 500

In [9]:
# training
captured_states = []
captured_epochs = []
loss_history = []

for epoch in range(epochs):
    optimizer.zero_grad()

    # capture hidden states
    hidden = model(x_train)
    captured_states.append(hidden.detach().cpu().numpy())

    out = model.output_linear(hidden)  # [B, S, 1]
    if not model.classification:
        out = out.squeeze(-1)  # [B, S] or [B]

    loss = criterion(out, y_train)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    captured_states.append(out.detach().numpy())
    captured_epochs.append(epoch)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1000x10 and 1x64)

In [ ]:
# Loss plotting
plt.figure(figsize=(8, 5))

plt.plot(loss_history, linewidth=2)
plt.title('Training Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
plt.close()

In [ ]:
# 3. Perform PCA on the Hidden States
pca = PCA(n_components=2)
hidden_states_2d = pca.fit_transform(hidden_states.numpy())

pc1 = hidden_states_2d[:, 0]
pc2 = hidden_states_2d[:, 1]

# 4. Visualize the Functional Surface over the Principal Components
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the network's learned surface (The CPWL hinges)
ax.scatter(pc1, pc2, network_predictions.numpy(),
           c='blue', alpha=0.5, label='Network Output (Learned Tropical Poly)', s=15)

# Plot the ground truth mapped to the same hidden PCA space
ax.scatter(pc1, pc2, y_train.numpy(),
           c='red', alpha=0.3, label='Ground Truth (Actual Tropical Poly)', s=15)

ax.set_title('Functional Surface projected onto Top 2 Principal Components')
ax.set_xlabel('Principal Component 1 (Hidden Space)')
ax.set_ylabel('Principal Component 2 (Hidden Space)')
ax.set_zlabel('Value / Output')
ax.legend()
plt.show()